In [24]:
import duckdb
con = duckdb.connect("hmda.db")


In [25]:
con.sql("DESCRIBE SELECT * FROM 'data/state_NC.csv'")

┌───────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│            column_name            │ column_type │  null   │   key   │ default │  extra  │
│              varchar              │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ activity_year                     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ lei                               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ derived_msa-md                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ state_code                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ county_code                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ census_tract                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ conforming_loan_limit             │ VARCHAR     │ YES     │ NULL    │ NULL    

In [26]:
import pandas as pd
pd.set_option('display.max_rows', None)
con.sql("DESCRIBE SELECT * FROM read_csv('data/state_NC.csv', ignore_errors=true)").df()

,column_name,column_type,null,key,default,extra
0,activity_year,BIGINT,YES,None,None,None
1,lei,VARCHAR,YES,None,None,None
2,derived_msa-md,BIGINT,YES,None,None,None
3,state_code,VARCHAR,YES,None,None,None
4,county_code,VARCHAR,YES,None,None,None
5,census_tract,VARCHAR,YES,None,None,None
6,conforming_loan_limit,VARCHAR,YES,None,None,None
7,derived_loan_product_type,VARCHAR,YES,None,None,None
8,derived_dwelling_category,VARCHAR,YES,None,None,None
9,derived_ethnicity,VARCHAR,YES,None,None,None


In [27]:
con.sql("""
CREATE OR REPLACE TABLE raw AS
SELECT * FROM read_csv('data/state_NC.csv', ignore_errors=true)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [28]:
con.sql("SELECT * FROM raw LIMIT 5")

┌───────────────┬──────────────────────┬────────────────┬────────────┬─────────────┬──────────────┬───────────────────────┬───────────────────────────┬──────────────────────────────────────┬─────────────────────────┬────────────────────┬───────────────────┬──────────────┬────────────────┬─────────────┬───────────┬──────────────┬─────────────┬──────────────────┬─────────────────────────┬────────────────────────────────┬─────────────┬─────────────────────┬───────────────┬─────────────┬──────────────┬──────────────────┬───────────────────────┬─────────────────────┬─────────────────┬────────────────┬───────────┬─────────────────────────┬───────────────────┬───────────────────────┬───────────────────────┬─────────────────┬──────────────────────────────┬────────────────┬─────────────────────┬────────────────┬─────────────────────────────────────────┬──────────────────────────────────────────┬─────────────┬──────────────────────────────┬─────────┬──────────────────────┬───────────────────────

In [29]:
con.sql("SELECT action_taken, COUNT(*) FROM raw GROUP BY 1 ORDER BY 2 DESC")

┌──────────────┬──────────────┐
│ action_taken │ count_star() │
│    int64     │    int64     │
├──────────────┼──────────────┤
│            1 │       259717 │
│            3 │        81317 │
│            4 │        62586 │
│            6 │        58717 │
│            5 │        32513 │
│            2 │        15405 │
│            8 │         2565 │
│            7 │          395 │
└──────────────┴──────────────┘

In [ ]:
cols = con.execute("DESCRIBE raw").df()["column_name"].tolist()
results = []
for col in cols:
    q = f"""
    SELECT '{col}' AS column_name,
    COUNT(*) - COUNT("{col}") AS null_count,
    COUNT(DISTINCT "{col}") AS distinct_values,
    FROM raw
"""
    results.append(con.execute(q).df())
profile = pd.concat(results, ignore_index=True)
print(profile)


                                 column_name  null_count  distinct_values  \
0                              activity_year           0                1   
1                                        lei           0             1200   
2                             derived_msa-md           0               17   
3                                 state_code           0                1   
4                                county_code           0              101   
5                               census_tract           0             2631   
6                      conforming_loan_limit           0                3   
7                  derived_loan_product_type           0                8   
8                  derived_dwelling_category           0                4   
9                          derived_ethnicity           0                5   
10                              derived_race           0                9   
11                               derived_sex           0                4   

In [37]:
con.sql("""
CREATE OR REPLACE TABLE loans AS
SELECT county_code, derived_ethnicity, derived_race, derived_sex, action_taken, purchaser_type, loan_type, loan_amount, interest_rate, loan_to_value_ratio,
income, applicant_credit_score_type, "applicant_ethnicity-1" AS applicant_ethnicity1, "applicant_ethnicity-2" AS applicant_ethnicity2, applicant_ethnicity_observed, "applicant_race-1" AS applicant_race1, applicant_race_observed,
applicant_sex, applicant_sex_observed, applicant_age, "denial_reason-1" AS denial_reason1, "denial_reason-2" AS denial_reason2, "denial_reason-3" AS denial_reason3
FROM raw
""")

con.sql("DESCRIBE SELECT * FROM loans").df()


,column_name,column_type,null,key,default,extra
0,county_code,VARCHAR,YES,None,None,None
1,derived_ethnicity,VARCHAR,YES,None,None,None
2,derived_race,VARCHAR,YES,None,None,None
3,derived_sex,VARCHAR,YES,None,None,None
4,action_taken,BIGINT,YES,None,None,None
5,purchaser_type,BIGINT,YES,None,None,None
6,loan_type,BIGINT,YES,None,None,None
7,loan_amount,DOUBLE,YES,None,None,None
8,interest_rate,VARCHAR,YES,None,None,None
9,loan_to_value_ratio,VARCHAR,YES,None,None,None
